# Classical ML Clickbait Classifier
**Models:** Multinomial Naive Bayes , Logistic Regression , Support Vector Machine
**Features:** TF-IDF (unigrams + bigrams) + 6 handcrafted features
**Data:** Pre-processed balanced dataset from `data_handling` step

---
## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import warnings
import json
import time
import os
import re

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    roc_auc_score
)
import joblib

print("All imports OK")

All imports OK


## 2. Load Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/NLPAOL_V3")

# Dataset all_agree
DATA_DIR = PROJECT_DIR / "data_1" / "annotated" / "combined" / "csv"
DATA_PATH = DATA_DIR / "all_agree.csv"

# Output classical model
OUTPUT_DIR = PROJECT_DIR / "outputs" / "classical_models_all_agree"
RESULTS_DIR = PROJECT_DIR / "outputs" / "classical_results_all_agree"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR :", PROJECT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("DATA_PATH   :", DATA_PATH)
print("DATA EXISTS :", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(f"all_agree.csv tidak ditemukan di: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Mounted at /content/drive
PROJECT_DIR : /content/drive/MyDrive/NLPAOL_V3
DATA_DIR    : /content/drive/MyDrive/NLPAOL_V3/data_1/annotated/combined/csv
DATA_PATH   : /content/drive/MyDrive/NLPAOL_V3/data_1/annotated/combined/csv/all_agree.csv
DATA EXISTS : True
Shape: (8613, 3)
Columns: ['title', 'label', 'label_score']


,title,label,label_score
0,"Masuk Radar Pilwalkot Medan, Menantu Jokowi Be...",non-clickbait,0
1,Malaysia Sudutkan RI: Isu Kabut Asap hingga In...,non-clickbait,0
2,Viral! Driver Ojol di Bekasi Antar Pesanan Mak...,clickbait,1
3,"Kemensos Salurkan Rp 7,3 M bagi Korban Kerusuh...",non-clickbait,0
4,MPR: Amandemen UUD 1945 Tak Akan Melebar ke Ma...,non-clickbait,0


## 3. EDA & Preprocessing
> **Dokumentasi lengkap transformasi dari `data_1` (raw) to `data` (cleaned)**
> Semua temuan di bawah menjelaskan mengapa dataset cleaned menghasilkan metrics lebih baik pada model klasikal.


In [3]:
# =======================================================
# EDA 1 - INSPEKSI AWAL DATASET
# all_agree.csv: subset CLICK-ID dengan stronger annotator agreement
# =======================================================

TEXT_COL = "title"
LABEL_COL = "label"
SCORE_COL = "label_score"

raw = pd.read_csv(DATA_PATH)  # DATA_PATH = ... / all_agree.csv

print("=" * 60)
print("RAW DATA OVERVIEW (all_agree.csv)")
print("=" * 60)
print(f"Shape   : {raw.shape[0]:,} rows × {raw.shape[1]} cols")
print(f"Columns : {raw.columns.tolist()}")
print()

if TEXT_COL not in raw.columns:
    raise ValueError(f"Kolom text '{TEXT_COL}' tidak ditemukan. Kolom tersedia: {raw.columns.tolist()}")

if LABEL_COL not in raw.columns:
    raise ValueError(f"Kolom label '{LABEL_COL}' tidak ditemukan. Kolom tersedia: {raw.columns.tolist()}")

print("[STEP 1] Missing values")
print(raw[[TEXT_COL, LABEL_COL]].isnull().sum())
print()

print("[STEP 2] Duplicate title check")
duplicate_count = raw.duplicated(subset=[TEXT_COL]).sum()
print(f"Duplicate titles : {duplicate_count:,}")
print()

print("[STEP 3] Short title check (< 3 words)")
word_count_raw = raw[TEXT_COL].astype(str).str.split().str.len()
short_count = (word_count_raw < 3).sum()
print(f"Short titles     : {short_count:,}")
print()

print("[STEP 4] Preview")
display(raw.head())


# Helper functions dipakai ulang pada EDA feature signal dan preprocessing final.
def clean_title(text):
    """
    Cleaning ringan untuk headline.
    Tujuannya bukan mengubah makna, hanya merapikan noise teknis.
    """
    text = str(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", "", text)

    # Remove zero-width / non-breaking spaces
    text = re.sub(r"[\u200b\u00a0\u200e\u200f]", " ", text)

    # Normalise curly quotes to straight quotes
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')

    # Normalise ellipsis unicode
    text = text.replace("\u2026", "...")

    # Strip pipe-separated source suffix, e.g. "Judul | detikNews"
    text = re.sub(r"\s*\|\s*.*$", "", text)

    # Strip hashtag symbol but keep the word
    text = re.sub(r"#(\w+)", r"\1", text)

    # Normalise colon attribution, e.g. "NAMA: kutipan" to "NAMA kutipan"
    text = re.sub(r"\b(\w{1,10}):\s+", r"\1 ", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_label(label):
    """
    Normalisasi label ke dua kelas standar:
    clickbait / non-clickbait.
    """
    label = str(label).strip().lower()

    if label in ["clickbait", "cb", "1", "true"]:
        return "clickbait"

    if label in ["non-clickbait", "nonclickbait", "non clickbait", "ncb", "0", "false"]:
        return "non-clickbait"

    return label


TRIGGERS = [
    r"\bviral\b", r"\bbikin\b", r"\bternyata\b", r"\bjangan\b",
    r"\bgila\b", r"\bcatat\b", r"\bsimak\b", r"\bbocoran\b",
    r"\bingat\b", r"\bwajib\b", r"\bdaftar\b", r"\brekomendasi\b",
    r"\btips\b", r"\bcara\b", r"\bfakta\b", r"\bratusan\b",
    r"\bribuan\b", r"\bmilyaran\b", r"\bshocking\b"
]

FEAT_COLS = [
    "char_len",
    "word_count",
    "has_exclaim",
    "has_question",
    "num_caps_words",
    "clickbait_trigger",
]


def engineer_features(df):
    """
    Feature engineering untuk model classical:
    TF-IDF tetap menjadi fitur utama, sedangkan kolom ini menjadi additional text features.
    """
    df = df.copy()

    df[TEXT_COL] = df[TEXT_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].apply(normalize_label)

    df["title_clean"] = df[TEXT_COL].apply(clean_title)
    df["char_len"] = df["title_clean"].str.len()
    df["word_count"] = df["title_clean"].str.split().str.len()
    df["has_exclaim"] = df["title_clean"].str.contains(r"!", regex=True).astype(int)
    df["has_question"] = df["title_clean"].str.contains(r"\?", regex=True).astype(int)
    df["num_caps_words"] = df["title_clean"].apply(
        lambda x: sum(1 for w in x.split() if w.isupper() and len(w) > 1)
    )
    df["clickbait_trigger"] = df["title_clean"].str.lower().str.contains(
        "|".join(TRIGGERS), regex=True
    ).astype(int)

    return df

RAW DATA OVERVIEW (all_agree.csv)
Shape   : 8,613 rows × 3 cols
Columns : ['title', 'label', 'label_score']

[STEP 1] Missing values
title    0
label    0
dtype: int64

[STEP 2] Duplicate title check
Duplicate titles : 11

[STEP 3] Short title check (< 3 words)
Short titles     : 2

[STEP 4] Preview


,title,label,label_score
0,"Masuk Radar Pilwalkot Medan, Menantu Jokowi Be...",non-clickbait,0
1,Malaysia Sudutkan RI: Isu Kabut Asap hingga In...,non-clickbait,0
2,Viral! Driver Ojol di Bekasi Antar Pesanan Mak...,clickbait,1
3,"Kemensos Salurkan Rp 7,3 M bagi Korban Kerusuh...",non-clickbait,0
4,MPR: Amandemen UUD 1945 Tak Akan Melebar ke Ma...,non-clickbait,0


### 4.1 Label Distribution

In [4]:
# =======================================================
# EDA 2 - LABEL DISTRIBUTION
# all_agree.csv
# =======================================================

print("Label distribution (all_agree.csv):")

vc = raw[LABEL_COL].apply(normalize_label).value_counts()

for label, count in vc.items():
    bar = "█" * int(count / 100)
    print(f"  {label:<15}: {count:,} ({count/len(raw)*100:.1f}%)  {bar}")

clickbait_count = vc.get("clickbait", 0)
non_clickbait_count = vc.get("non-clickbait", 0)

if non_clickbait_count > 0:
    imbalance_ratio = clickbait_count / non_clickbait_count
    print(f"\n  Class imbalance ratio: {imbalance_ratio:.3f}")
else:
    print("\n  Class imbalance ratio: cannot calculate, non-clickbait = 0")

print("  to all_agree masih memiliki imbalance, sehingga class_weight='balanced' dipakai pada LR & SVM.")

Label distribution (all_agree.csv):
  non-clickbait  : 5,297 (61.5%)  ████████████████████████████████████████████████████
  clickbait      : 3,316 (38.5%)  █████████████████████████████████

  Class imbalance ratio: 0.626
  to all_agree masih memiliki imbalance, sehingga class_weight='balanced' dipakai pada LR & SVM.


### 4.2 Title Length Distribution

In [5]:
# =======================================================
# EDA 3 - TITLE LENGTH ANALYSIS
# all_agree.csv
# =======================================================

print("Title word-count stats (all_agree.csv):")

tmp_len = raw.copy()
tmp_len[LABEL_COL] = tmp_len[LABEL_COL].apply(normalize_label)
tmp_len["word_count"] = tmp_len[TEXT_COL].astype(str).str.split().str.len()

wc = tmp_len["word_count"]

print(f"  mean  = {wc.mean():.1f}")
print(f"  median= {wc.median():.0f}")
print(f"  min   = {wc.min()}")
print(f"  max   = {wc.max()}")
print(f"  p90   = {wc.quantile(0.90):.0f}")
print(f"  p95   = {wc.quantile(0.95):.0f}")
print(f"  p99   = {wc.quantile(0.99):.0f}")
print()

print("By class:")
class_means = {}
for lbl in ["clickbait", "non-clickbait"]:
    sub = tmp_len[tmp_len[LABEL_COL] == lbl]["word_count"]
    class_means[lbl] = sub.mean()
    print(f"  {lbl:<15}: mean={sub.mean():.1f}  median={sub.median():.0f}  max={sub.max()}")

diff = class_means.get("clickbait", 0) - class_means.get("non-clickbait", 0)
print()
print(f"to Rata-rata panjang judul clickbait berbeda sekitar {diff:.1f} kata dari non-clickbait.")
print("  Karena headline relatif pendek, TF-IDF unigram/bigram masih cocok untuk baseline classical.")

Title word-count stats (all_agree.csv):
  mean  = 9.6
  median= 9
  min   = 2
  max   = 19
  p90   = 13
  p95   = 14
  p99   = 15

By class:
  clickbait      : mean=10.6  median=10  max=18
  non-clickbait  : mean=9.1  median=9  max=19

to Rata-rata panjang judul clickbait berbeda sekitar 1.5 kata dari non-clickbait.
  Karena headline relatif pendek, TF-IDF unigram/bigram masih cocok untuk baseline classical.


### 4.3 Feature Signal Analysis

In [6]:
# =======================================================
# EDA 4 - ADDITIONAL TEXT FEATURE SIGNAL STRENGTH
# Tunjukkan kenapa setiap fitur tambahan berguna untuk klasifikasi classical
# =======================================================

eda_df = engineer_features(raw)

cb_df = eda_df[eda_df[LABEL_COL] == "clickbait"]
nc_df = eda_df[eda_df[LABEL_COL] == "non-clickbait"]

print(f"{'Feature':<22} {'CB mean':>9} {'NC mean':>9} {'Ratio':>7}  Insight")
print("-" * 85)

features_info = [
    ("char_len",          "panjang karakter headline"),
    ("word_count",        "jumlah kata headline"),
    ("has_exclaim",       "indikator tanda seru"),
    ("has_question",      "indikator tanda tanya / curiosity gap"),
    ("clickbait_trigger", "indikator kata trigger clickbait"),
    ("num_caps_words",    "jumlah kata kapital / akronim"),
]

for feat, insight in features_info:
    cb_m = cb_df[feat].mean()
    nc_m = nc_df[feat].mean()
    ratio = cb_m / (nc_m + 1e-9)
    print(f"  {feat:<20} {cb_m:>9.3f} {nc_m:>9.3f} {ratio:>7.1f}x  {insight}")

print()
print("to TF-IDF menjadi fitur utama, sedangkan kolom di atas menjadi additional text features.")
print("  Fitur tambahan ini membantu model classical menangkap sinyal sederhana seperti panjang judul, tanda baca, dan trigger words.")

Feature                  CB mean   NC mean   Ratio  Insight
-------------------------------------------------------------------------------------
  char_len                69.229    60.196     1.2x  panjang karakter headline
  word_count              10.572     9.063     1.2x  jumlah kata headline
  has_exclaim              0.078     0.001    82.1x  indikator tanda seru
  has_question             0.165     0.000 165259348.6x  indikator tanda tanya / curiosity gap
  clickbait_trigger        0.183     0.028     6.5x  indikator kata trigger clickbait
  num_caps_words           0.340     0.666     0.5x  jumlah kata kapital / akronim

to TF-IDF menjadi fitur utama, sedangkan kolom di atas menjadi additional text features.
  Fitur tambahan ini membantu model classical menangkap sinyal sederhana seperti panjang judul, tanda baca, dan trigger words.


### 3.4 Preprocessing Pipeline (data_1 to data)

In [7]:
# =======================================================
# EDA 5 - FULL PREPROCESSING PIPELINE SUMMARY
# Build the cleaned dataframe from all_agree.csv for classical models.
# =======================================================

print("Preprocessing pipeline for all_agree.csv")
print("=" * 60)

before_rows = len(raw)

cleaned = engineer_features(raw)

# Keep only valid labels
valid_labels = ["clickbait", "non-clickbait"]
before_valid = len(cleaned)
cleaned = cleaned[cleaned[LABEL_COL].isin(valid_labels)].copy()
removed_invalid = before_valid - len(cleaned)

# Remove empty titles after cleaning
before_empty = len(cleaned)
cleaned = cleaned[cleaned["title_clean"].notna() & (cleaned["title_clean"].str.strip() != "")].copy()
removed_empty = before_empty - len(cleaned)

# Remove duplicate cleaned titles to reduce leakage risk
before_dedup = len(cleaned)
cleaned = cleaned.drop_duplicates(subset=["title_clean"]).copy()
removed_duplicates = before_dedup - len(cleaned)

# Remove titles that are too short to carry enough meaning
before_short = len(cleaned)
cleaned = cleaned[cleaned["word_count"] >= 3].copy()
removed_short = before_short - len(cleaned)

cleaned = cleaned.reset_index(drop=True)

print(f"Rows before preprocessing : {before_rows:,}")
print(f"Removed invalid labels    : {removed_invalid:,}")
print(f"Removed empty titles      : {removed_empty:,}")
print(f"Removed duplicate titles  : {removed_duplicates:,}")
print(f"Removed short titles      : {removed_short:,}")
print(f"Rows after preprocessing  : {len(cleaned):,}")
print()

print("Columns used later:")
print("  Text column              : title_clean")
print("  Label column             : label")
print("  Additional text features :", FEAT_COLS)
print()

sample_raw = pd.DataFrame({
    TEXT_COL: [
        "KPK Tanggapi DPR: Penegakan Hukum Tak Boleh Terikat Komitmen Politik",
        "ACT Inisiasi Gerakan Nasional #IndonesiaDermawan",
        "Viral! Driver Ojol di Bekasi Antar Pesanan Makanan Pakai Sepeda",
        "Berita Terkini | detikNews",
    ],
    LABEL_COL: ["non-clickbait", "non-clickbait", "clickbait", "non-clickbait"],
    SCORE_COL: [0, 0, 1, 0],
})

sample_clean = engineer_features(sample_raw)

print("Pipeline verification:")
for _, row in sample_clean[[TEXT_COL, "title_clean", "has_exclaim", "has_question", "clickbait_trigger"]].iterrows():
    print(f"  RAW  : {row[TEXT_COL]}")
    print(f"  CLEAN: {row['title_clean']}")
    print(f"         exclaim={row['has_exclaim']}  question={row['has_question']}  trigger={row['clickbait_trigger']}")
    print()

print("clean_title() and engineer_features() were used to build the cleaned dataframe.")

Preprocessing pipeline for all_agree.csv
Rows before preprocessing : 8,613
Removed invalid labels    : 0
Removed empty titles      : 0
Removed duplicate titles  : 13
Removed short titles      : 2
Rows after preprocessing  : 8,598

Columns used later:
  Text column              : title_clean
  Label column             : label
  Additional text features : ['char_len', 'word_count', 'has_exclaim', 'has_question', 'num_caps_words', 'clickbait_trigger']

Pipeline verification:
  RAW  : KPK Tanggapi DPR: Penegakan Hukum Tak Boleh Terikat Komitmen Politik
  CLEAN: KPK Tanggapi DPR Penegakan Hukum Tak Boleh Terikat Komitmen Politik
         exclaim=0  question=0  trigger=0

  RAW  : ACT Inisiasi Gerakan Nasional #IndonesiaDermawan
  CLEAN: ACT Inisiasi Gerakan Nasional IndonesiaDermawan
         exclaim=0  question=0  trigger=0

  RAW  : Viral! Driver Ojol di Bekasi Antar Pesanan Makanan Pakai Sepeda
  CLEAN: Viral! Driver Ojol di Bekasi Antar Pesanan Makanan Pakai Sepeda
         exclaim=1  q

### 3.5 Data Split & Leakage Check

In [8]:
# =======================================================
# EDA 6 - SPLIT VALIDATION (zero leakage)
# Create splits from the cleaned all_agree dataframe.
# =======================================================

# 80% train, 20% temporary
train, temp = train_test_split(
    cleaned,
    test_size=0.20,
    random_state=42,
    stratify=cleaned[LABEL_COL]
)

# 10% validation, 10% test
val, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=42,
    stratify=temp[LABEL_COL]
)

train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

# Final classical model training uses train + validation data.
# The validation split is still used earlier for hyperparameter tuning.
train_full = pd.concat([train, val], ignore_index=True)

print("Split summary (stratified, random_state=42):")
total = len(train) + len(val) + len(test)
print(f"  Train : {len(train):,} ({len(train)/total*100:.1f}%)")
print(f"  Val   : {len(val):,}  ({len(val)/total*100:.1f}%)")
print(f"  Test  : {len(test):,}  ({len(test)/total*100:.1f}%)")
print(f"  Train+Val for final fit : {len(train_full):,}")
print()

print("Label distribution per split:")
for name, split_df in [("Train", train), ("Val", val), ("Test", test)]:
    vc2 = split_df[LABEL_COL].value_counts()
    cb = vc2.get("clickbait", 0)
    nc = vc2.get("non-clickbait", 0)

    print(
        f"  {name:5s}: "
        f"CB={cb:,} ({cb/len(split_df)*100:.1f}%)  "
        f"NCB={nc:,} ({nc/len(split_df)*100:.1f}%)"
    )

print()

# Use title_clean to check whether any duplicate headline appears across splits.
train_titles = set(train["title_clean"])
val_titles   = set(val["title_clean"])
test_titles  = set(test["title_clean"])

o1 = len(train_titles & val_titles)
o2 = len(train_titles & test_titles)
o3 = len(val_titles & test_titles)

print("Leakage check based on title_clean:")
print(f"  Train vs Val  : {o1} overlap  {'OK' if o1 == 0 else 'LEAKAGE DETECTED'}")
print(f"  Train vs Test : {o2} overlap  {'OK' if o2 == 0 else 'LEAKAGE DETECTED'}")
print(f"  Val   vs Test : {o3} overlap  {'OK' if o3 == 0 else 'LEAKAGE DETECTED'}")
print()

print("Split dibuat setelah duplicate removal, sehingga risiko data leakage dari headline duplikat lebih kecil.")

Split summary (stratified, random_state=42):
  Train : 6,878 (80.0%)
  Val   : 860  (10.0%)
  Test  : 860  (10.0%)
  Train+Val for final fit : 7,738

Label distribution per split:
  Train: CB=2,649 (38.5%)  NCB=4,229 (61.5%)
  Val  : CB=331 (38.5%)  NCB=529 (61.5%)
  Test : CB=331 (38.5%)  NCB=529 (61.5%)

Leakage check based on title_clean:
  Train vs Val  : 0 overlap  OK
  Train vs Test : 0 overlap  OK
  Val   vs Test : 0 overlap  OK

Split dibuat setelah duplicate removal, sehingga risiko data leakage dari headline duplikat lebih kecil.


## 4. Feature Extraction
### 4.1 TF-IDF Vectorizer (unigrams + bigrams)

In [9]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30_000,
    sublinear_tf=True,       # log(1+tf) - helps with short texts
    min_df=2,
    strip_accents="unicode",
    analyzer="word",
)

X_train_tfidf = tfidf.fit_transform(train_full["title_clean"].fillna(""))
X_test_tfidf  = tfidf.transform(test["title_clean"].fillna(""))

# For validation (hyperparameter tuning)
X_tr_t = tfidf.transform(train["title_clean"].fillna(""))
X_val_t = tfidf.transform(val["title_clean"].fillna(""))

print(f"Vocabulary size        : {len(tfidf.vocabulary_):,}")
print(f"TF-IDF matrix (train)  : {X_train_tfidf.shape}")

Vocabulary size        : 13,667
TF-IDF matrix (train)  : (7738, 13667)


### 4.2 Additional Text Features

In [10]:
scaler = MinMaxScaler()
X_train_hand = scaler.fit_transform(train_full[FEAT_COLS].values)
X_test_hand  = scaler.transform(test[FEAT_COLS].values)

# Val split for tuning
X_tr_h  = scaler.transform(train[FEAT_COLS].values)
X_val_h = scaler.transform(val[FEAT_COLS].values)

print(f"Handcrafted features   : {len(FEAT_COLS)}")
print(f"Feature names          : {FEAT_COLS}")

Handcrafted features   : 6
Feature names          : ['char_len', 'word_count', 'has_exclaim', 'has_question', 'num_caps_words', 'clickbait_trigger']


### 4.3 Combined Feature Matrix (TF-IDF + Additional Text)

In [11]:
# Final matrices (train+val combined for fitting, test for evaluation)
X_train_combined = sp.hstack([X_train_tfidf, sp.csr_matrix(X_train_hand)], format="csr")
X_test_combined  = sp.hstack([X_test_tfidf,  sp.csr_matrix(X_test_hand)],  format="csr")

# Matrices for hyperparameter tuning (train only to val)
X_tr  = sp.hstack([X_tr_t,  sp.csr_matrix(X_tr_h)],  format="csr")
X_val = sp.hstack([X_val_t, sp.csr_matrix(X_val_h)],  format="csr")

# Labels
y_train = train_full["label"].map({"clickbait": 1, "non-clickbait": 0}).values
y_test  = test["label"].map({"clickbait": 1, "non-clickbait": 0}).values
y_tr    = train["label"].map({"clickbait": 1, "non-clickbait": 0}).values
y_val   = val["label"].map({"clickbait": 1, "non-clickbait": 0}).values

print(f"Combined train matrix  : {X_train_combined.shape}")
print(f"Combined test matrix   : {X_test_combined.shape}")

Combined train matrix  : (7738, 13673)
Combined test matrix   : (860, 13673)


## 5. Evaluation Helper

In [12]:
def evaluate(name, model, X_test, y_test, proba=True):
    t0 = time.time()
    y_pred = model.predict(X_test)
    inf_time = (time.time() - t0) / len(y_test) * 1000  # ms/sample

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_test, y_pred, average="macro", zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    auc = None
    if proba:
        try:
            y_prob = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        except Exception:
            pass

    print(f"{'-'*50}")
    print(f"  {name}")
    print(f"{'-'*50}")
    print(f"  Accuracy   : {acc:.4f}  {'PASS' if acc >= 0.85 else 'CHECK'} (target >= 0.85)")
    print(f"  Precision  : {prec:.4f}  (macro)")
    print(f"  Recall     : {rec:.4f}  (macro)")
    print(f"  F1         : {f1:.4f}  {'PASS' if f1 >= 0.80 else 'CHECK'} (target >= 0.80)")
    if auc:
        print(f"  AUC-ROC    : {auc:.4f}")
    print(f"  Inf. time  : {inf_time:.4f} ms/sample")
    print(f"\n  Confusion Matrix (rows=actual, cols=predicted):")
    print(f"                 Pred Non-CB   Pred CB")
    print(f"  Actual Non-CB     {cm[0,0]:>5}      {cm[0,1]:>5}")
    print(f"  Actual CB         {cm[1,0]:>5}      {cm[1,1]:>5}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred,
          target_names=["non-clickbait", "clickbait"], digits=4))

    return {
        "model": name,
        "accuracy": round(acc, 4),
        "precision_macro": round(prec, 4),
        "recall_macro": round(rec, 4),
        "f1_macro": round(f1, 4),
        "auc_roc": round(auc, 4) if auc else None,
        "inference_ms_per_sample": round(inf_time, 4),
        "confusion_matrix": cm.tolist(),
    }

print("evaluate() helper defined")

evaluate() helper defined


## 6. Model 1 - Multinomial Naive Bayes
### 8.1 Hyperparameter Tuning (alpha)

In [13]:
print("Tuning alpha on validation set...\n")
best_alpha, best_f1 = 1.0, 0.0
tune_results = []

for alpha in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0]:
    mnb_tmp = MultinomialNB(alpha=alpha)
    mnb_tmp.fit(X_tr, y_tr)
    f1_val = f1_score(y_val, mnb_tmp.predict(X_val), average="macro")
    tune_results.append((alpha, f1_val))
    marker = " ← best" if f1_val > best_f1 else ""
    print(f"  alpha={alpha:.2f}  val_f1={f1_val:.4f}{marker}")
    if f1_val > best_f1:
        best_f1, best_alpha = f1_val, alpha

print(f"\nBest alpha : {best_alpha}  (val F1 = {best_f1:.4f})")

Tuning alpha on validation set...

  alpha=0.01  val_f1=0.8560 ← best
  alpha=0.05  val_f1=0.8633 ← best
  alpha=0.10  val_f1=0.8678 ← best
  alpha=0.30  val_f1=0.8667
  alpha=0.50  val_f1=0.8677
  alpha=1.00  val_f1=0.8697 ← best
  alpha=2.00  val_f1=0.8666

Best alpha : 1.0  (val F1 = 0.8697)


### 8.2 Train Final MNB

In [14]:
mnb = MultinomialNB(alpha=best_alpha)
t0 = time.time()
mnb.fit(X_train_combined, y_train)
print(f"Train time : {time.time()-t0:.2f}s")

joblib.dump(mnb, f"{OUTPUT_DIR}/mnb_model.pkl")
print(f"Model saved to {OUTPUT_DIR}/mnb_model.pkl")

Train time : 0.01s
Model saved to /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_all_agree/mnb_model.pkl


### 8.3 Evaluate MNB on Test Set

In [15]:
res_mnb = evaluate("Multinomial Naive Bayes", mnb, X_test_combined, y_test)

--------------------------------------------------
  Multinomial Naive Bayes
--------------------------------------------------
  Accuracy   : 0.8721  PASS (target >= 0.85)
  Precision  : 0.8808  (macro)
  Recall     : 0.8485  (macro)
  F1         : 0.8597  PASS (target >= 0.80)
  AUC-ROC    : 0.9401
  Inf. time  : 0.0016 ms/sample

  Confusion Matrix (rows=actual, cols=predicted):
                 Pred Non-CB   Pred CB
  Actual Non-CB       503         26
  Actual CB            84        247

  Classification Report:
               precision    recall  f1-score   support

non-clickbait     0.8569    0.9509    0.9014       529
    clickbait     0.9048    0.7462    0.8179       331

     accuracy                         0.8721       860
    macro avg     0.8808    0.8485    0.8597       860
 weighted avg     0.8753    0.8721    0.8693       860



## 7. Model 2 - Logistic Regression
### 8.1 Hyperparameter Tuning (C)

In [16]:
print("Tuning C on validation set...\n")
best_C_lr, best_f1 = 1.0, 0.0

for C in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    lr_tmp = LogisticRegression(C=C, solver="saga", max_iter=1000,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    lr_tmp.fit(X_tr, y_tr)
    f1_val = f1_score(y_val, lr_tmp.predict(X_val), average="macro")
    marker = " ← best" if f1_val > best_f1 else ""
    print(f"  C={C:<5}  val_f1={f1_val:.4f}{marker}")
    if f1_val > best_f1:
        best_f1, best_C_lr = f1_val, C

print(f"\nBest C : {best_C_lr}  (val F1 = {best_f1:.4f})")

Tuning C on validation set...

  C=0.01   val_f1=0.8024 ← best
  C=0.1    val_f1=0.8524 ← best
  C=0.5    val_f1=0.8927 ← best
  C=1.0    val_f1=0.8999 ← best
  C=5.0    val_f1=0.9215 ← best
  C=10.0   val_f1=0.9201

Best C : 5.0  (val F1 = 0.9215)


### 8.2 Train Final LR

In [17]:
lr = LogisticRegression(
    C=best_C_lr, solver="saga", max_iter=1000,
    class_weight="balanced", random_state=42, n_jobs=-1
)
t0 = time.time()
lr.fit(X_train_combined, y_train)
print(f"Train time : {time.time()-t0:.2f}s")

joblib.dump(lr, f"{OUTPUT_DIR}/lr_model.pkl")
print(f"Model saved to {OUTPUT_DIR}/lr_model.pkl")

Train time : 0.49s
Model saved to /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_all_agree/lr_model.pkl


### 8.3 Evaluate LR on Test Set

In [18]:
res_lr = evaluate("Logistic Regression", lr, X_test_combined, y_test)

--------------------------------------------------
  Logistic Regression
--------------------------------------------------
  Accuracy   : 0.9081  PASS (target >= 0.85)
  Precision  : 0.9109  (macro)
  Recall     : 0.8942  (macro)
  F1         : 0.9011  PASS (target >= 0.80)
  AUC-ROC    : 0.9614
  Inf. time  : 0.0009 ms/sample

  Confusion Matrix (rows=actual, cols=predicted):
                 Pred Non-CB   Pred CB
  Actual Non-CB       505         24
  Actual CB            55        276

  Classification Report:
               precision    recall  f1-score   support

non-clickbait     0.9018    0.9546    0.9275       529
    clickbait     0.9200    0.8338    0.8748       331

     accuracy                         0.9081       860
    macro avg     0.9109    0.8942    0.9011       860
 weighted avg     0.9088    0.9081    0.9072       860



## 8. Model 3 - Support Vector Machine
### 8.1 Hyperparameter Tuning (C)

In [19]:
print("Tuning C on validation set...\n")
best_C_svm, best_f1 = 1.0, 0.0

for C in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]:
    svm_tmp = LinearSVC(C=C, class_weight="balanced", max_iter=2000,
                         random_state=42, dual=True)
    svm_tmp.fit(X_tr, y_tr)
    f1_val = f1_score(y_val, svm_tmp.predict(X_val), average="macro")
    marker = " ← best" if f1_val > best_f1 else ""
    print(f"  C={C:<4}  val_f1={f1_val:.4f}{marker}")
    if f1_val > best_f1:
        best_f1, best_C_svm = f1_val, C

print(f"\nBest C : {best_C_svm}  (val F1 = {best_f1:.4f})")

Tuning C on validation set...

  C=0.01  val_f1=0.8531 ← best
  C=0.1   val_f1=0.9072 ← best
  C=0.5   val_f1=0.9213 ← best
  C=1.0   val_f1=0.9161
  C=2.0   val_f1=0.9164
  C=5.0   val_f1=0.9140

Best C : 0.5  (val F1 = 0.9213)


### 8.2 Train Final SVM

In [20]:
# CalibratedClassifierCV wraps LinearSVC to provide predict_proba (needed for AUC)
svm_base = LinearSVC(C=best_C_svm, class_weight="balanced",
                      max_iter=2000, random_state=42, dual=True)
svm = CalibratedClassifierCV(svm_base, cv=3)

t0 = time.time()
svm.fit(X_train_combined, y_train)
print(f"Train time : {time.time()-t0:.2f}s")

joblib.dump(svm, f"{OUTPUT_DIR}/svm_model.pkl")
print(f"Model saved to {OUTPUT_DIR}/svm_model.pkl")

Train time : 0.13s
Model saved to /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_all_agree/svm_model.pkl


### 8.3 Evaluate SVM on Test Set

In [21]:
res_svm = evaluate("SVM (LinearSVC + Calibrated)", svm, X_test_combined, y_test)

--------------------------------------------------
  SVM (LinearSVC + Calibrated)
--------------------------------------------------
  Accuracy   : 0.9116  PASS (target >= 0.85)
  Precision  : 0.9165  (macro)
  Recall     : 0.8965  (macro)
  F1         : 0.9046  PASS (target >= 0.80)
  AUC-ROC    : 0.9611
  Inf. time  : 0.0082 ms/sample

  Confusion Matrix (rows=actual, cols=predicted):
                 Pred Non-CB   Pred CB
  Actual Non-CB       509         20
  Actual CB            56        275

  Classification Report:
               precision    recall  f1-score   support

non-clickbait     0.9009    0.9622    0.9305       529
    clickbait     0.9322    0.8308    0.8786       331

     accuracy                         0.9116       860
    macro avg     0.9165    0.8965    0.9046       860
 weighted avg     0.9129    0.9116    0.9105       860



## 9. Save Artifacts & Results Summary

In [22]:
results = [res_mnb, res_lr, res_svm]
with open(f"{OUTPUT_DIR}/classical_results.json", "w") as f:
    json.dump(results, f, indent=2)

joblib.dump(tfidf,  f"{OUTPUT_DIR}/tfidf_vectorizer.pkl")
joblib.dump(scaler, f"{OUTPUT_DIR}/feature_scaler.pkl")

print("Saved:")
print(f"  {OUTPUT_DIR}/classical_results.json")
print(f"  {OUTPUT_DIR}/tfidf_vectorizer.pkl")
print(f"  {OUTPUT_DIR}/feature_scaler.pkl")

Saved:
  /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_all_agree/classical_results.json
  /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_all_agree/tfidf_vectorizer.pkl
  /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_all_agree/feature_scaler.pkl


## 10. Comparison Table

In [23]:
print(f"{'Model':<35} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7} {'ms/s':>8}")
print("-" * 78)
for r in results:
    auc_str = f"{r['auc_roc']:.4f}" if r["auc_roc"] else "  N/A "
    acc_flag = " PASS" if r["accuracy"] >= 0.85 else "  "
    f1_flag  = " PASS" if r["f1_macro"] >= 0.80  else "  "
    print(f"{r['model']:<35} {r['accuracy']:>7.4f}{acc_flag} "
          f"{r['precision_macro']:>7.4f} {r['recall_macro']:>7.4f} "
          f"{r['f1_macro']:>7.4f}{f1_flag} {auc_str:>7} "
          f"{r['inference_ms_per_sample']:>8.4f}")
print()
print("PASS = meets target  |  Target: Accuracy >= 0.85, F1 >= 0.80")

Model                                   Acc    Prec     Rec      F1     AUC     ms/s
------------------------------------------------------------------------------
Multinomial Naive Bayes              0.8721 PASS  0.8808  0.8485  0.8597 PASS  0.9401   0.0016
Logistic Regression                  0.9081 PASS  0.9109  0.8942  0.9011 PASS  0.9614   0.0009
SVM (LinearSVC + Calibrated)         0.9116 PASS  0.9165  0.8965  0.9046 PASS  0.9611   0.0082

PASS = meets target  |  Target: Accuracy >= 0.85, F1 >= 0.80
